In [1]:
import numpy as np
import pandas as pd
import os
import glob
import itertools
import scanpy as sc
import natsort
import json

from scroutines import basicu

import seaborn as sns

In [2]:
# ddir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/cheng21_cell_scrna/organized'
# outdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results/GABAARs'
# !mkdir -p $outdir

In [3]:
# def get_cond_from_biosample(sample):
#     """
#     """
#     cond = sample[:-1]
#     if not cond.endswith('DR'): 
#         cond = cond+'NR'
        
#     return cond

In [4]:
%%time
f  = "../../data/v1_multiome/superdupermegaRNA_hasraw_multiome_IT.h5ad"
adata_all = sc.read(f)

CPU times: user 2.44 s, sys: 19.3 s, total: 21.8 s
Wall time: 1min 52s


In [5]:
adata_all.obs['cond'] = adata_all.obs['Age']
adata_all.obs['cond'].unique()

['P6', 'P8', 'P10', 'P12', 'P12DR', ..., 'P14DR', 'P17', 'P17DR', 'P21', 'P21DR']
Length: 11
Categories (11, object): ['P6', 'P8', 'P10', 'P12', ..., 'P17', 'P17DR', 'P21', 'P21DR']

In [6]:
unq_samples = natsort.natsorted(adata_all.obs['Sample'].unique())
print(len(unq_samples))
print(unq_samples) 

unq_conds   = natsort.natsorted(adata_all.obs['cond'].unique())
print(len(unq_conds))
print(unq_conds)

unq_types   = natsort.natsorted(adata_all.obs['Subclass'].unique())
print(len(unq_types))
print(unq_types)

25
['P6a', 'P6b', 'P6c', 'P8a', 'P8b', 'P8c', 'P10a', 'P10b', 'P12DRa', 'P12DRb', 'P12a', 'P12b', 'P12c', 'P14DRa', 'P14DRb', 'P14a', 'P14b', 'P17DRa', 'P17DRb', 'P17a', 'P17b', 'P21DRa', 'P21DRb', 'P21a', 'P21b']
11
['P6', 'P8', 'P10', 'P12', 'P12DR', 'P14', 'P14DR', 'P17', 'P17DR', 'P21', 'P21DR']
4
['L2/3', 'L4', 'L5IT', 'L6IT']


In [7]:
adata_all.X.data

array([7., 1., 1., ..., 1., 4., 1.], dtype=float32)

In [8]:
%%time

typecol = 'dpt_qbin_n15'

adata_all.obs[typecol] = -1

num_bins = 15

adata_dpt_dict = {}
for cond in unq_conds:
    
    f = f'/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/multiome_dpt_{cond}.h5ad'
    
    adata_dpt = sc.read(f, backed='r')
    
    # add bins
    xm = (1/2 + np.arange(0,num_bins,1))*1/num_bins
    adata_dpt.obs[typecol] = pd.qcut(adata_dpt.obs['dpt'], num_bins, labels=False)
    adata_all.obs[typecol].update(adata_dpt.obs[typecol])

    adata_dpt_dict[cond] = adata_dpt
    
    # break

CPU times: user 1.7 s, sys: 4.16 s, total: 5.86 s
Wall time: 22.8 s


In [9]:
adata_all.obs[typecol].unique()

array([ 7, 10, 13,  5, 14,  9,  6,  0,  1,  4,  8, 11,  3, 12,  2])

In [10]:
### check size

for i, cond_name in enumerate(unq_conds):
    print(cond_name)
    
    adata = adata_all[adata_all.obs['cond']==cond_name]
    
    # check gene 
    if i == 0:
        genes = adata.var.index.values
    else:
        assert np.all(genes == adata.var.index.values)
        
    # check sample
    sample_lbls = adata.obs['Sample'].values
    unq_samples = np.unique(sample_lbls)
    
    # check type
    unq_types = np.unique(adata.obs[typecol].values)

    # check size
    nr, nc, ng =  len(unq_samples), len(unq_types), len(adata.var)  #  rep, subclass, gene 
    print(nr, nc, ng)
    

P6
3 15 16567
P8
3 15 16567
P10
2 15 16567
P12
3 15 16567
P12DR
2 15 16567
P14
2 15 16567
P14DR
2 15 16567
P17
2 15 16567
P17DR
2 15 16567
P21
2 15 16567
P21DR
2 15 16567


In [11]:
nc, ng

(15, 16567)

In [12]:
### read in the file and prep

nf = len(unq_conds)
nr = 3 # dummy for some
bigtensor = np.zeros((nf, nr, nc, ng)) # conditions, replicates, types, genes

for i, cond_name in enumerate(unq_conds):
    print(cond_name)
    
    adata = adata_all[adata_all.obs['cond']==cond_name]
        
    ### sum over counts 
    # by sample
    sample_lbls = adata.obs['Sample'].values
    unq_samples = np.unique(sample_lbls)

    # by type
    type_lbls = adata.obs[typecol].values

    nr, nc, ng =  len(unq_samples), len(unq_types), len(adata.var)  #  rep, subclass, gene 

    for j, this_samp in enumerate(unq_samples):
        for k, this_type in enumerate(unq_types):
            selector = ((sample_lbls==this_samp) & (type_lbls==this_type))
            bigtensor[i,j,k] = np.sum(np.array(adata[selector].X.todense()), axis=0) # raw reads sum over all cells
    
    if j < 2:
        bigtensor[i,2] = np.nan 
            
### CPM  - per cond, sample and subclass
# normalize it as log2(1+CPM)
bigtensor = (bigtensor/np.sum(bigtensor, axis=-1, keepdims=True))*1e6
bigtensor = np.log2(1+bigtensor) 

P6
P8
P10
P12
P12DR
P14
P14DR
P17
P17DR
P21
P21DR


In [14]:
bigtensor.shape

(11, 3, 15, 16567)

In [17]:
unq_types
genes.shape

(16567,)

In [32]:
output      = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/res/bigtensor_subclass_it_15bin.npy' 
output_meta = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/res/bigtensor_subclass_it_15bin.json' 


In [33]:
np.save(output, bigtensor)

meta = {
    'size': bigtensor.shape,
    
    'dim0': list(unq_conds),
    'dim1': ['a', 'b', 'c'],
    'dim2': list(unq_types),
    'dim3': list(genes),
}

with open(output_meta, 'w') as fp:
    json.dump(meta, fp)

In [34]:
bigtensor = np.load(output)
bigtensor.shape

(11, 3, 18, 16572)

In [35]:
with open(output_meta, 'r') as fp:
    meta = json.load(fp)